# Fine-Tuning IndicF5 on Tamil Voice Corpus (TPU / Colab / Kaggle)

This notebook guides you through the process of fine-tuning **IndicF5** (AI4Bharat's multilingual Text-to-Speech model) on the **Tamil Voice Corpus** (`ragunath-ravi/TamilVoiceCorpus`) using Google Cloud TPU (or Colab/Kaggle TPU).

## 🚨 1. Preventing Overfitting

Since you are fine-tuning a ~400M parameter model on a Tamil corpus (which has ~10K - 100K utterances), full fine-tuning can lead to **overfitting** (where the model memorizes the speaker's voice but loses its general pronunciation capabilities or distorts speech quality). To prevent overfitting, we implement the following strategies:

1. **Adapter-Style Fine-Tuning (`--freeze-backbone`) [RECOMMENDED]**:
   This freezes the core DiT (Diffusion Transformer) layers and only trains the text embeddings, input/output projections, and layer norms. This drastically reduces the number of trainable parameters, prevents catastrophic forgetting of multilingual capabilities, and speeds up training.
2. **Exponential Moving Average (EMA)**:
   The training loop maintains an EMA of the model's weights (with a decay of `0.9999`). Checkpoints are saved with EMA weights, which provide smoother speech synthesis and are highly resistant to overfitting.
3. **Lower Learning Rate**:
   We use a learning rate of `1e-5` for full fine-tuning, or `5e-5` for adapter-style tuning. A small learning rate ensures that the pre-trained weights drift slowly and stay regularized.
4. **Early Stopping**:
   Fine-tuning typically converges in **5 to 15 epochs**. You do not need the default 50 epochs.

## 🛠️ 2. Environment Setup (TPU / XLA)

Run the following cell to install the necessary libraries for PyTorch XLA (TPU) and audio processing.

In [ ]:
# Install PyTorch XLA (if not preinstalled on Colab TPU)
!pip install -q accelerate datasets soundfile librosa vocos huggingface_hub

# Check if TPU is detected
import torch
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print(f'SUCCESS: TPU detected! Device = {device}')
except ImportError:
    print('WARNING: PyTorch XLA (torch_xla) not installed. Training will fall back to GPU/CPU.')

## 📂 3. Clone Repository

Clone your newly created GitHub repository. (Replace the URL below with your actual repository URL).

In [ ]:
# Replace with your new repository URL:
!git clone https://github.com/Ragu-123/indicf5.git
%cd indicf5

## 📝 4. Setup Custom Data Preprocessor & Trainer

If you cloned the repository containing our updates, the files `src/f5_tts/model/trainer.py` and `src/indicf5_finetune/prepare_tamil_data.py` are already updated and ready to use.

We can run them directly!

## 🔑 5. Authenticate to Hugging Face

Since the `TamilVoiceCorpus` dataset is private/gated, you must authenticate using your Hugging Face API Token.

In [ ]:
import os
from huggingface_hub import login

try:
    # Try pulling the HF_TOKEN from Kaggle Secrets if running in Kaggle
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        login(token=hf_token)
        print('SUCCESS: Logged in using Kaggle Secrets HF_TOKEN!')
    else:
        raise ValueError('HF_TOKEN secret is empty')
except Exception as e:
    print(f'Could not load token from Kaggle Secrets ({e}). Falling back to interactive/manual login.')
    login()

## 📦 6. Prepare Tamil Voice Corpus Data

Run the dataset preparation script. You can choose to train on `PureVox` (recommended, cleaner speech) or `AmbientVox` (larger, noisier speech).

In [ ]:
# Run data preparation for PureVox subset (15.5k training samples)
!python src/indicf5_finetune/prepare_tamil_data.py --data-dir ./data_tamil --subset PureVox --split train

## 🚀 7. Run Fine-Tuning on TPU (PyTorch XLA)

To launch training on TPUs via Hugging Face `accelerate`, we configure it to use XLA device placement. 

> [!IMPORTANT]
> **Preventing Overfitting**: By default, we use `--freeze-backbone` to run **Adapter-style fine-tuning**. This is highly recommended because it freezes the DiT transformer backbone and only trains the embeddings and output projections. This keeps the pre-trained Tamil pronunciations intact and keeps the training fast and regularized.

In [ ]:
# Launch training with accelerate on TPU
# We set --freeze-backbone to prevent overfitting!
!accelerate launch --tpu \
    -m indicf5_finetune.train \
    --data-dir ./data_tamil \
    --freeze-backbone \
    --epochs 15 \
    --lr 5e-5 \
    --batch-size 38400 \
    --grad-accum 2

## 🔊 8. Generate Audio (Inference / Evaluation)

Once training finishes, you can evaluate the fine-tuned model by synthesizing Tamil/English audio using a reference voice clip.

In [ ]:
# Synthesize audio from the checkpoints folder
# Ensure you have a reference voice WAV file and transcript to guide style cloning!
!python -m indicf5_finetune.evaluate \
    --checkpoint-dir ./checkpoints \
    --ref-audio ./voices/TAM_F_HAPPY_00001.wav \
    --ref-text 'your reference transcript here'